# Setting things up


First we mount the file system to access the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Then we define a bunch of auxiliary functions, including the training loop!

In [ ]:
from math import pi as PI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import gc
import os

import tensorflow as tf
from keras.optimizers import AdamW
from keras.models import Model

from keras.losses import BinaryCrossentropy
from keras.losses import binary_crossentropy

MMU = 105.658 # (MeV) muon mass
MN = 939.56542052 # (MeV) neutron mass
MP = 938.27208943 # (MeV) proton mass
MNUC = (MN + MP)/2 # (MeV) averaged proton and neutron mass
E_BPN = 4.0 # (MeV) binding energy per nucleon (aprox) lower boundary


def load_nuwro_events(
        dataset_name, n_events=None, from_end=False, shuffle=False
    ):
    """
    Returns the 'csv' file containing NuWro generated events as a dataframe

    Arguments:
        dataset_name : string or list of strings
            path/file of the 'csv' to read

    Optional arguments
        n_events : int
            number of events that will be loaded. If not specified, the whole
            dataset is returned
    """
    # load dataset # TODO redefine for tensors
    if isinstance(dataset_name, list):
        df_list = []
        for data_name in dataset_name:
            tmp_data = pd.read_csv(data_name)
            df_list.append(tmp_data)
        data = pd.concat(df_list, axis=0, ignore_index=True)
    elif isinstance(dataset_name, str):
        data = pd.read_csv(dataset_name)

    if shuffle:
        data = data.sample(frac=1).reset_index(drop=True)

    if n_events is not None:
        if from_end:
            data = data[-n_events:]
        else:
            data = data[:n_events]

    kin_vars = ["Emu", "thetamu", "Enu"]
    # make sure we are working with floats
    data[kin_vars] = data[kin_vars].astype('float32')

    return data


def events_for_training(nuwro_data):
    # manipulate variables so they all lie on the interval (-1,1)
    training_data = normalize_dataframe(nuwro_data, lepmass=MMU)
    Enu = np.reshape(training_data["Enu"], [training_data["Enu"].shape[0],1])

    return [training_data[["Emu", "thetamu"]], Enu]


def normalize_Enu(Enu, min_Enu=None, max_Enu=None):
    # (min, max) gets mapped to (-1,1)
    max_Enu = max_Enu - min_Enu
    vEnu = Enu - min_Enu
    vEnu = 2*vEnu/max_Enu - 1

    return vEnu


def normalize_Emu(Emu, Enu, lepmass=MMU, Ebpn=E_BPN):
    DeltaE = Enu - lepmass - Ebpn
    groundedE = Emu - lepmass
    Emu = groundedE/DeltaE
    Emu = 2*np.sqrt(1 - Emu) - 1

    return Emu


def normalize_thetamu(thetamu):
    theta = thetamu/PI
    theta = 2*np.sqrt(theta) - 1

    return theta


def normalize_dataframe(
        data, Enu=None, lepmass=None, Ebpn=E_BPN,
    ):
    """
    Returns a normalized Emu and thetamu so they lie in the interval [-1,1].
    Distributions skewness is reduced by obtaining square roots of normalized
    distributions

    Arguments:
        data : dataframe
            dataframe containing the kinematic variables of each event. By
            default it is assumed the dataframe contains the event neutrino
            energy 'Enu' and the lepton mass 'lepmass'
    Optional Arguments:
        Enu : double, np.array
            The Neutrino energy of the events. Should contain as many event
            entries as the dataframe in case of np.array
        lepmass : double, np.array
            The resultant lepton mass of the events. Should contain as many
            event entries as the dataframe in case of np.array
        Ebpn : double
            Gap between the maximum possible Emu value and Enu in CCQE
            scattering
    """
    data = data.copy()
    if Enu is None:
        Enu = data["Enu"]
    if lepmass is None:
        lepmass =  data["lepmass"]

    data["Emu"] = normalize_Emu(data["Emu"], Enu, lepmass, E_BPN)
    data["thetamu"] = normalize_thetamu(data["thetamu"])
    data["Enu"] = normalize_Enu(Enu, 110, 10000)

    return data


def log_loss(BCE):
    def loss(y_true, y_pred, sample_weight=None):
        return  - BCE(1.0-y_true, y_pred) + BCE(y_true, y_pred)

    return loss


@tf.function
def train_step_dis(batch_nuw, label_nuw, label_dnn, d_model, g_model,
        dnn_Enu, latent_points, dis_opt, loss_func):
    dnn_Enu = tf.cast(dnn_Enu, tf.float32)

    # First we train the discriminator
    # train discriminator on true

    with tf.GradientTape() as tape_nuw:
        pred_nuw = d_model(batch_nuw, training=True)
        loss_nuw = loss_func(label_nuw, pred_nuw)

    grads_nuw = tape_nuw.gradient(loss_nuw, d_model.trainable_variables)
    dis_opt.apply_gradients(zip(grads_nuw, d_model.trainable_variables))

    # train discriminator on generated sample
    with tf.GradientTape() as tape_dnn:
        batch_dnn = [generator([latent_points, dnn_Enu], training=True), dnn_Enu]

        label_dnn = tf.zeros_like(batch_dnn[1])
        pred_dnn = d_model(batch_dnn, training=True)
        loss_dnn = loss_func(label_dnn, pred_dnn)

    grads_dnn = tape_dnn.gradient(loss_dnn, d_model.trainable_variables)
    dis_opt.apply_gradients(zip(grads_dnn, d_model.trainable_variables))

    return loss_nuw, loss_dnn


@tf.function(reduce_retracing=True)
def train_step_gen(label_gan, d_model, g_model,
        batch_size, gan_Enu, latent_points, gen_opt, loss_func
    ):
    gan_Enu = tf.cast(gan_Enu, dtype=tf.float32)
    latent_points = tf.cast(latent_points, tf.float32)

    # Then we train the generator
    with tf.GradientTape() as tape_gen:
        # we need to keep the sample generation inside GradientTape()!
        batch_muon = g_model([latent_points, gan_Enu], training=True)

        label_gan = tf.ones_like(gan_Enu)
        batch_gan = [batch_muon, gan_Enu]

        pred_gan = d_model(batch_gan, training=True)
        loss_gan = loss_func(label_gan, pred_gan)

    grads_gan = tape_gen.gradient(loss_gan, g_model.trainable_variables)
    gen_opt.apply_gradients(zip(grads_gan, g_model.trainable_variables))

    return loss_gan


def train(g_model, d_model, train_dataset, train_size, latent_dim,
        n_epochs=10, batch_size=100, lr_gen=0.0001, lr_dis=0.001,
        use_amsgrad=False):
    half_batch = int(batch_size/2)
    bat_per_epo = int(train_size/batch_size)
    twice_batch = 2 * batch_size

    bce = BinaryCrossentropy(from_logits=True)
    bceh = log_loss(bce)
    gen_opt = AdamW(
            learning_rate=lr_gen,
            beta_1=0.5, beta_2=0.9,
            amsgrad=use_amsgrad
        )
    dis_opt = AdamW(
            learning_rate=lr_dis,
            beta_1=0.5, beta_2=0.9,
            amsgrad=use_amsgrad
        )

    print("There are {} batches per epoch".format(bat_per_epo))

    # get NuWro and generator labels
    label_nuw = tf.ones([batch_size, 1])
    label_dnn = tf.zeros([batch_size, 1])
    # create inverted labels for the fake samples
    label_gan = tf.ones([2*batch_size, 1])

    d_loss1 = d_loss2 = g_loss = 0
    # manually enumerate epochs
    min_chi2_loss = 200
    loss_xsec = 500
    for epoch in range(n_epochs):
        data_batches = train_dataset.shuffle(
                buffer_size=int(train_size/4)
            ).batch(batch_size)
        step = 0

        # enumerate batches over the training set
        for batch_nuw in data_batches:
            latent_points, dnn_Enu = get_gen_input(
                batch_size, latent_dim)

            d_loss1, d_loss2 = train_step_dis(
                batch_nuw, label_nuw, label_dnn, d_model, g_model,
                dnn_Enu, latent_points, dis_opt, bce
            )

            if step % 5 == 0:# and epoch > 0:
                # Train the generator
                batch_size_gen = 2*batch_size

                latent_points, gan_Enu = get_gen_input(batch_size_gen,
                    latent_dim)

                gan_Enu = tf.cast(gan_Enu, dtype=tf.float32)

                g_loss = train_step_gen(
                    label_gan, d_model, g_model,
                    2*batch_size, gan_Enu, latent_points,
                    gen_opt, bceh
                )


            # summarize loss on this batch
            #if (epoch + 1) % 1 == 0 and step + 1 == bat_per_epo:
            if (step + 1)%100 == 0:

                progress = "t>{}, {}/{}".format(epoch+1, step+1, bat_per_epo)

                # stats to print in terminal
                nuw_stats = "l_nuw = {:.4f}".format(float(d_loss1))
                gen_stats = (
                    "l_gen = {:.4f}, l_gan = {:.4f}".format(
                        float(d_loss2), float(g_loss)
                    )
                )

                message = "{}, {}, {}".format(
                        progress, nuw_stats, gen_stats
                    )
                print(message)

            step += 1
        # end of batch loop
    # end of epoch loop

    return g_model


# Task 1
* Write the correct paths to the dataset files.
* Write the correct paths to generator and discriminator.

> Tip: It is better to use absolute paths.

In [ ]:
# load nuwro dataset
# TODO: Replace by correct filepath
datasets = [
      "",
      "",
      ""
    ]

train_dataframe = load_nuwro_events(datasets, n_events=None, shuffle=True)
# events_for_training normalizes the sample
train_events, train_E = events_for_training(train_dataframe)

# Now we load both the generator and discriminator
# TODO: Replace by correct filepath
gen_name = ""
dis_name = ""

print("Loading generator", gen_name)
generator = tf.keras.models.load_model(gen_name)
discriminator = tf.keras.models.load_model(dis_name)


# Task 2
We need to get inputs for the GAN. It should receive "n_examples" points (vectors) from the latent space of size "latent_dim" and conditions uniformly distributed between (-1,1)

* Write a function that outputs the latent points and the conditions

 > Tip: Use tf.random.normal() and tf.random.uniform() functions. First dimension corresponds to the number of examples "n" and the second to the number of random elements "r" -> [n, r]






In [ ]:
def get_gen_input(n_examples, latent_dim):
    # generate points in latent space
    latent_points = # Field to fill
    dnn_Enu = # Field to fill

    return latent_points, dnn_Enu

In [ ]:
# We can compare the output of the base model and compare to the dataset
latent_dim = 50

input = get_gen_input(500000, latent_dim)
gen_sample = generator(input).numpy()

plt.hist(gen_sample[:,0], bins=50, histtype='step', density=True, label='generator')
plt.hist(train_events['Emu'].to_numpy(), bins=50, histtype='step', density=True, label='data')
plt.legend(loc="upper left")
plt.xlabel("Emu")
plt.ylabel("Probability density")
plt.show()
plt.close()

plt.hist(gen_sample[:,1], bins=50, histtype='step', density=True, label='generator')
plt.hist(train_events['thetamu'].to_numpy(), bins=50, histtype='step', density=True, label='data')
plt.legend(loc="upper right")
plt.xlabel("theta")
plt.ylabel("Probability density")
plt.show()
plt.close()



# Task 3
* Write two histograms in 2D.
  * One histogram for the dataset distribution.
  * Anoher histogram for the genrated data.

> Tip: Use plt.hist2d().

In [ ]:
# Write your code here to generate the histograms

# Task 4
* Change the learning rates, number of epochs, batch size, turn on and off "amsgrad", etc.
* Train the network.

> Tips:
>* Use small learning rates.
>* Try to set generator learning rate smaller by a factor of ten respect to discriminator learning rate.
>* Start training only 1 or 2 epochs.
>* Try to balance batch size and number of epochs.
>* "asmgrad" stabilizes training, but might stop optimization before getting to the true optimal value.



In [ ]:
train_size = train_events.shape[0]

print("Training events to process:", train_size)
train_dataset = tf.data.Dataset.from_tensor_slices((train_events, train_E))
n_epochs = # Field to fill
batch_size = # Field to fill
lr_gen = # Field to fill
lr_dis = # Field to fill
use_amsgrad = # Field to fill

# train the network
generator = train(
    generator, discriminator, train_dataset, train_size, latent_dim,
    n_epochs=n_epochs, batch_size=batch_size, lr_gen=lr_gen, lr_dis=lr_dis,
    use_amsgrad=use_amsgrad
)


Training events to process: 970000


# Task 5
* With the trained network recreate the histograms of the produced variables and compared with the dataset.

In [ ]:
# test the new generator
gen_sample = generator(input).numpy()

# Make new plots comparing the generator and the dataset


NameError: name 'generator' is not defined

# Task 6
* Write functions to take the variables used for training into true neutrino and muon kinematic space $(E_\nu, E_\mu, \theta)$.
* Using $(E_\nu, E_\mu, \theta)$, get the hadronic system invariant mass $W$ and make the histograms comparing the dataset and the generated sample.

> Tip: Check at the definitions of the normalization functions in the section "Setting things up".



In [ ]:
# Define your functions here and make the histograms
